# PI-CAI Model Demo Notebook — Phase 1 (modèle, sans XAI)

Pipeline en 10 cellules, sur le cas de test officiel puis tous les cas démo téléchargés.
Pré-requis : `.venv` installé (`pip install -r requirements.txt` à la racine du
sous-projet) et un lancement de `scripts/download_or_prepare_data.py`.

> ⚠️ Sous-projet `prostate_model_demo/` — phase 1 = modèle seulement ; pas de
> XAI ici (voir `README.md` du sous-projet).

## Cellule 1 — imports + chemin du projet

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import SimpleITK as sitk

from src.config import CASE_LEVEL_THRESHOLD, OUTPUTS_DIR
from src.data_loader import (
    load_case, list_available_cases,
    load_marksheet, get_case_ground_truth, load_lesion_annotation,
)
from src.preprocessing import preprocess_images, make_tensor
from src.inference import get_device, load_models
from src.evaluation import lesion_localization_case, case_level_metrics

print('cas disponibles :', list_available_cases())

## Cellule 2 — matériel disponible (GPU ?)

In [ ]:
DEVICE, note = get_device()
print('device :', DEVICE, '|', note)

## Cellule 3 — charger le cas + vérité terrain + annotation d'expert
Comme dans `scripts/run_demo.py` : `load_case` enrichi avec la marksheet
officielle (`case_level_csPCa`, `case_ISUP`) et le contour d'expert
(`resampled/` multi-classes ou `Pooch25/` binaire).

In [ ]:
def prepare_case(case_id):
    case = load_case(case_id)
    mark = load_marksheet()
    case.clinical.update(get_case_ground_truth(mark, case.case_id))
    case.lesion_annotation = load_lesion_annotation(case.case_id)
    return case

case = prepare_case('10032_1000032')
t2w = sitk.GetArrayFromImage(case.images['t2w'])
print('case', case.case_id, '| source:', case.source)
print('t2w shape (z,y,x):', t2w.shape)
print('t2w spacing:', case.images['t2w'].GetSpacing())
print('GT (marksheet): case_csPCa =', case.clinical['case_level_csPCa'],
      '| case_ISUP =', case.clinical.get('case_ISUP'))

## Cellule 4 — prétraitement (pipeline officiel)
Alignement COM → crop/pad (20,256,256) → resample (3.0,0.5,0.5) mm → z-score
par modalité → tenseur `(1, 3, 20, 256, 256)`.

In [ ]:
scans = [case.images[m] for m in ('t2w', 'adc', 'hbv')]
input_batch = preprocess_images(scans)
tensor = make_tensor(input_batch, DEVICE)
print('tensor:', tensor.shape, '→', DEVICE)

## Cellule 5 — charger les 5 checkpoints (ensemble officiel)

In [ ]:
models, folds = load_models(device=DEVICE)
print('folds:', folds, '→', len(models), 'modèles chargés sur', DEVICE)

## Cellule 6 — inférence (TTA flip + sigmoid + moyenne 5 folds + lissage σ=1.5)

In [ ]:
from src.inference import predict

heatmap = predict(tensor, models, DEVICE)
print('heatmap brute:', heatmap.shape, 'max =', round(float(heatmap.max()), 4))

## Cellule 7 — post-traitement (grille native T2W)
Carte de détection (blobs « lésions candidates ») + probabilité patient = max(carte).

In [ ]:
from src.inference import run_inference

result = run_inference(case, device=DEVICE)
print('probabilité de cas :', round(result.case_level_likelihood, 4))
print('carte de détection :', result.detection_map.shape,
      'max =', round(float(result.detection_map.max()), 4))
print('décision           :',
      'csPCa détecté' if result.case_level_likelihood >= CASE_LEVEL_THRESHOLD
      else 'pas de csPCa', f'(seuil {CASE_LEVEL_THRESHOLD})')

## Cellule 8 — vérité terrain + métriques micro
`case_level_metrics` attend une LISTE de (gt, score) — même format que
`run_demo`. IoU 3D vs contour d'expert quand il existe (annotations
`resampled/` multi-classes ou `Pooch25/` binaires).

In [ ]:
metrics = case_level_metrics([(case.clinical, result.case_level_likelihood)])
print('seuil 0.5  :', metrics['y_score'], 'vs y_true', metrics['y_true'])

if case.lesion_annotation is not None:
    lesion = lesion_localization_case(result.detection_map, case.lesion_annotation)
    print("IoU 3D détection vs contour expert:", round(lesion['iou_detection_vs_gt'], 3),
          "| GT a une lésion:", lesion['gt_has_lesion'])
else:
    print("Pas d'annotation d'expert pour ce cas.")

## Cellule 9 — figure (IRM native + carte de détection + contour expert vs détection)

In [ ]:
from src.visualization import case_figure

fig_path = case_figure(
    case, result,
    save_path=OUTPUTS_DIR / 'figures' / f'nb_{case.case_id}.png',
    title=f'PI-CAI case {case.case_id} — probabilité '
          f'{result.case_level_likelihood*100:.1f}%')
print('figure écrite :', fig_path)

## Cellule 10 — répéter sur tous les cas disponibles

In [ ]:
rows = []
for cid in list_available_cases():
    c = prepare_case(cid)
    r = run_inference(c, device=DEVICE)
    row = {'case_id': cid,
           'probability': round(r.case_level_likelihood, 3),
           'threshold_OK': r.case_level_likelihood >= CASE_LEVEL_THRESHOLD,
           'gt_csPCa': c.clinical.get('case_level_csPCa')}
    if c.lesion_annotation is not None:
        row['iou'] = round(lesion_localization_case(r.detection_map,
                             c.lesion_annotation)['iou_detection_vs_gt'], 3)
    rows.append(row)

pd.DataFrame(rows).sort_values('probability', ascending=False)

---
### Étape suivante : phase 2 (XAI) — hors périmètre de cette démo
Grad-CAM / Integrated Gradients / occlusion, à comparer avec
`result.detection_map` et `case.lesion_annotation`. Voir le README phase 1.